In [ ]:
-- Set the context for this session
USE ROLE ACCOUNTADMIN;

In [ ]:
CREATE OR REPLACE DATABASE FINANCE_APP_DB;
USE DATABASE FINANCE_APP_DB;
CREATE OR REPLACE SCHEMA FINANCE_APP_SCHEMA;
USE SCHEMA FINANCE_APP_SCHEMA;

In [ ]:
CREATE OR REPLACE TABLE SP500_PRICE (
    DATE DATE,
    CLOSE FLOAT,
    HIGH FLOAT,
    LOW FLOAT,
    OPEN FLOAT,
    VOLUME NUMBER(38,0)
);

CREATE OR REPLACE TABLE SAMSUNG_PRICE (
    DATE DATE,
    CLOSE FLOAT,
    HIGH FLOAT,
    LOW FLOAT,
    OPEN FLOAT,
    VOLUME NUMBER(38,0)
);

In [ ]:
-- Create the Native App Package
DROP APPLICATION PACKAGE finance_app_package; -- This is needed only if exercise is repeated
CREATE APPLICATION PACKAGE finance_app_package;
CREATE OR REPLACE SCHEMA finance_app_schema;
CREATE OR REPLACE STAGE finance_app_stage 
    DIRECTORY = ( ENABLE = true ) 
    COMMENT = '';

In [ ]:
CREATE SCHEMA shared_content_schema;
USE SCHEMA shared_content_schema;
CREATE OR REPLACE VIEW SP500_PRICE AS SELECT * FROM finance_app_db.finance_app_schema.sp500_price;
CREATE OR REPLACE VIEW SAMSUNG_PRICE AS SELECT * FROM finance_app_db.finance_app_schema.samsung_price;

In [ ]:
GRANT REFERENCE_USAGE ON DATABASE finance_app_db TO SHARE IN APPLICATION PACKAGE finance_app_package;

GRANT USAGE ON SCHEMA shared_content_schema TO SHARE IN APPLICATION PACKAGE finance_app_package;

GRANT SELECT ON VIEW sp500_price TO SHARE IN APPLICATION PACKAGE finance_app_package;
GRANT SELECT ON VIEW samsung_price TO SHARE IN APPLICATION PACKAGE finance_app_package;

In [ ]:
ALTER APPLICATION PACKAGE finance_app_package
  REGISTER VERSION V1
  USING '@finance_app_package.finance_app_schema.finance_app_stage/src';

In [ ]:
COPY INTO finance_app_db.finance_app_schema.sp500_price FROM @finance_app_package.finance_app_schema.finance_app_stage/data/S&P500_portfolio_data.csv
FILE_FORMAT = (TYPE = 'CSV' SKIP_HEADER = 3);

COPY INTO finance_app_db.finance_app_schema.samsung_price FROM @finance_app_package.finance_app_schema.finance_app_stage/data/SAMSUNG_portfolio_data.csv
FILE_FORMAT = (TYPE = 'CSV' SKIP_HEADER = 3);

In [ ]:
DROP APPLICATION  finance_app_v1;
CREATE APPLICATION finance_app_v1 FROM APPLICATION PACKAGE finance_app_package USING VERSION V1 PATCH 0;
SHOW APPLICATIONS;